# 강화학습

#### 라이브러리

In [ ]:
import numpy as np
import random
import time
import matplotlib.pyplot as plt

from IPython.display import clear_output

#### 미로 정의

In [ ]:
MAZE_SIZE = 401

maze = np.ones(
    (MAZE_SIZE, MAZE_SIZE),
    dtype=np.uint8
)

def generate_maze(start_y=1, start_x=1):

    stack = [(start_y, start_x)]
    maze[start_y, start_x] = 0

    while stack:

        y, x = stack[-1]

        directions = [
            (-2, 0),
            (2, 0),
            (0, -2),
            (0, 2)
        ]

        random.shuffle(directions)

        found = False

        for dy, dx in directions:

            ny = y + dy
            nx = x + dx

            if (
                1 <= ny < MAZE_SIZE - 1
                and 1 <= nx < MAZE_SIZE - 1
                and maze[ny, nx] == 1
            ):

                maze[
                    y + dy // 2,
                    x + dx // 2
                ] = 0

                maze[ny, nx] = 0

                stack.append((ny, nx))

                found = True
                break

        if not found:
            stack.pop()


generate_maze()

start = (1, 1)
goal = (MAZE_SIZE - 2, MAZE_SIZE - 2)

print(maze.shape)

#### 미로 확인

In [ ]:
plt.figure(figsize=(12, 12))

plt.imshow(
    maze,
    cmap="gray_r",
    interpolation="nearest"
)

plt.scatter(
    start[1],
    start[0],
    s=50,
    label="Start"
)

plt.scatter(
    goal[1],
    goal[0],
    s=100,
    marker="*",
    label="Goal"
)

plt.legend()

plt.show()

#### 행동 정의

In [ ]:
actions = [
    (-1, 0),  # 위
    (1, 0),   # 아래
    (0, -1),  # 왼쪽
    (0, 1)    # 오른쪽
]

num_actions = len(actions)

#### Q-Table 만들기
- 점수를 저장하는 거임

In [ ]:
Q = np.zeros(
    (MAZE_SIZE, MAZE_SIZE, num_actions),
    dtype=np.float32
)

print(Q.shape)
print(f"{Q.nbytes / 1024 / 1024:.2f} MB")

#### 보상함수
- 목표 도착 +100
- 벽 충돌 -10
- 맵 밖 -10
- 한 칸 이동 -1

In [ ]:
def step(state, action):

    y, x = state

    dy, dx = actions[action]

    ny = y + dy
    nx = x + dx

    # 벽 또는 범위 밖
    if (
        ny < 0
        or ny >= MAZE_SIZE
        or nx < 0
        or nx >= MAZE_SIZE
        or maze[ny, nx] == 1
    ):
        return state, -10, False

    next_state = (ny, nx)

    if next_state == goal:
        return next_state, 1000, True

    return next_state, -1, False

In [ ]:
%matplotlib widget

In [ ]:
fig, ax = plt.subplots(
    figsize=(12, 12)
)

ax.imshow(
    maze,
    cmap="gray_r",
    interpolation="nearest"
)

ax.scatter(
    start[1],
    start[0],
    s=50,
    label="Start"
)

ax.scatter(
    goal[1],
    goal[0],
    s=100,
    marker="*",
    label="Goal"
)

# AI
agent_dot = ax.scatter(
    start[1],
    start[0],
    s=80
)

# 이동 경로
path_line, = ax.plot(
    [],
    [],
    linewidth=1
)

title = ax.set_title("Ready")

ax.legend()

plt.show()

#### 시각화 함수

In [1]:
def draw_maze(path=None, episode=None):
    clear_output(wait=True)

    plt.figure(figsize=(12, 12))
    plt.imshow(maze, cmap="gray_r", interpolation="nearest")

    plt.scatter(
        start[1], start[0],
        s=100,
        marker="o",
        label="Start"
    )

    plt.scatter(
        goal[1], goal[0],
        s=150,
        marker="*",
        label="Goal"
    )

    if path:
        ys = [p[0] for p in path]
        xs = [p[1] for p in path]

        plt.plot(xs, ys, linewidth=1)

    if episode is not None:
        plt.title(f"Episode {episode}")

    plt.legend()
    plt.show()

#### 학습

In [2]:
alpha = 0.2
gamma = 0.99

epsilon = 1.0
epsilon_decay = 0.995
epsilon_min = 0.05

episodes = 5000

for episode in range(1, episodes + 1):

    state = start

    path = [state]

    total_reward = 0

    # 큰 미로니까 넉넉하게
    max_steps = 200000

    for step_count in range(max_steps):

        y, x = state

        # =====================
        # 행동 선택
        # =====================

        if random.random() < epsilon:

            action = random.randint(
                0,
                num_actions - 1
            )

        else:

            action = np.argmax(
                Q[y, x]
            )

        # =====================
        # 환경 이동
        # =====================

        next_state, reward, done = step(
            state,
            action
        )

        ny, nx = next_state

        # =====================
        # Q-Learning
        # =====================

        current_q = Q[
            y,
            x,
            action
        ]

        max_next_q = np.max(
            Q[ny, nx]
        )

        Q[y, x, action] = (
            current_q
            + alpha
            * (
                reward
                + gamma * max_next_q
                - current_q
            )
        )

        state = next_state

        path.append(state)

        total_reward += reward

        # =====================
        # 실시간 화면 갱신
        # =====================

        if step_count % 100 == 0:

            agent_dot.set_offsets([
                [state[1], state[0]]
            ])

            # 최근 5000칸만 표시
            recent_path = path[-5000:]

            xs = [p[1] for p in recent_path]
            ys = [p[0] for p in recent_path]

            path_line.set_data(
                xs,
                ys
            )

            title.set_text(
                f"Episode {episode} | "
                f"Step {step_count} | "
                f"Reward {total_reward} | "
                f"Epsilon {epsilon:.3f}"
            )

            fig.canvas.draw_idle()
            fig.canvas.flush_events()

        if done:

            print(
                f"GOAL! "
                f"Episode={episode}, "
                f"Steps={step_count}"
            )

            break

    epsilon = max(
        epsilon_min,
        epsilon * epsilon_decay
    )

NameError: name 'start' is not defined

#### 학습 후 탐색 없이 100% AI가 배운 최적 행동만 따라가게하기

In [ ]:
state = start
best_path = [state]

for _ in range(200000):

    y, x = state

    action = np.argmax(Q[y, x])

    next_state, reward, done = step(
        state,
        action
    )

    state = next_state
    best_path.append(state)

    if done:
        print("GOAL!")
        break

else:
    print("Goal 못 찾음")

draw_maze(
    best_path,
    episode="Final"
)